# Phase-5: Clustering into Priority Tiers (London-final-light)

**Goal:** Group London LSOAs into actionable priority tiers based on policing demand.

Clustering features = `risk_score_scaled` + the validated model features (which exclude `stop_search_rate` and `seasonal_volatility`).

- **5a. Optimal k selection:** K-Means with silhouette scores (k=2 to 6)
- **5b. GMM comparison:** Gaussian Mixture Models with AIC/BIC
- **5c. Cluster profiling** + choropleth map

**Reads:** `london-final-light/outputs/phase4/` · **Writes:** `london-final-light/outputs/phase5/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.preprocessing import MinMaxScaler
import geopandas as gpd

print('Imports OK')

In [ ]:
# Loading Dataset
BASE = Path('Dataset Path')
P4   = BASE / 'london-final-light' / 'outputs' / 'phase4'
OUT  = BASE / 'london-final-light' / 'outputs' / 'phase5'
OUT.mkdir(parents=True, exist_ok=True)
SHP_DIR = BASE / 'data' / 'LB_shp'

TIER_COLOURS = {1: '#D62728', 2: '#FF7F0E', 3: '#2CA02C', 4: '#1F77B4', 5: '#9467BD', 6: '#BCBD22'}
print('Output folder:', OUT)

## Section-1: Load Data

In [ ]:
risk = pd.read_parquet(P4 / 'phase4_risk_scores.parquet')
print(f'Loaded: {risk.shape}')
print(f'Columns: {risk.columns.tolist()}')

# Model features = everything that is not metadata / score bookkeeping
NON_FEAT = {'lsoa21cd','lsoa21nm','lad22nm','crime_count','risk_score',
            'risk_score_scaled','nb_predicted','nb_ratio'}
MODEL_FEATS = [c for c in risk.columns if c not in NON_FEAT]
CLUSTER_FEATURES = ['risk_score_scaled'] + MODEL_FEATS
print(f'\nClustering features: {CLUSTER_FEATURES}')

In [ ]:
scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(risk[CLUSTER_FEATURES]), columns=CLUSTER_FEATURES, index=risk.index)
print('Clustering matrix shape:', X.shape)
print('Feature ranges (should all be 0-1):')
print(X.agg(['min','max']).round(3))

## 5a. Optimal k-Selection: K-Means + Silhouette

In [ ]:
print('Testing K-Means for k=2 to 6...')
k_range = range(2, 7)
inertias = []; silhouettes = []; km_models = {}
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels)
    inertias.append(km.inertia_); silhouettes.append(sil); km_models[k] = (km, labels)
    print(f'  k={k}  inertia={km.inertia_:.1f}  silhouette={sil:.4f}')
best_k_sil = list(k_range)[int(np.argmax(silhouettes))]
print(f'\nBest k by silhouette: {best_k_sil}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(list(k_range), inertias, 'o-', color='#378ADD', linewidth=2)
axes[0].set_xlabel('Number of clusters (k)', fontsize=11)
axes[0].set_ylabel('Inertia', fontsize=11)
axes[0].set_title('Elbow method — london-final-light', fontsize=12)
axes[0].spines[['top','right']].set_visible(False)
colours = ['#D62728' if k == best_k_sil else '#378ADD' for k in k_range]
bars = axes[1].bar(list(k_range), silhouettes, color=colours, edgecolor='white')
axes[1].set_xlabel('Number of clusters (k)', fontsize=11)
axes[1].set_ylabel('Silhouette score', fontsize=11)
axes[1].set_title('Silhouette score by k (red = best)', fontsize=12)
axes[1].spines[['top','right']].set_visible(False)
for bar, sil in zip(bars, silhouettes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{sil:.3f}', ha='center', va='bottom', fontsize=9)
plt.suptitle('K-Means cluster selection — london-final-light', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUT / '5a_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5a_silhouette.png')

## 5b. GMM Comparison: AIC/BIC

In [ ]:
print('Fitting Gaussian Mixture Models for k=2 to 6...')
aics = []; bics = []; gmm_models = {}
for k in k_range:
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=5, max_iter=300)
    gmm.fit(X)
    aics.append(gmm.aic(X)); bics.append(gmm.bic(X)); gmm_models[k] = gmm
    print(f'  k={k}  AIC={gmm.aic(X):.1f}  BIC={gmm.bic(X):.1f}')
best_k_aic = list(k_range)[int(np.argmin(aics))]
best_k_bic = list(k_range)[int(np.argmin(bics))]
print(f'\nBest k by AIC: {best_k_aic}')
print(f'Best k by BIC: {best_k_bic}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_range), aics, 'o-', color='#378ADD', linewidth=2, label='AIC')
ax.plot(list(k_range), bics, 's--', color='#D85A30', linewidth=2, label='BIC')
ax.axvline(best_k_bic, color='#D85A30', alpha=0.3, linestyle=':', label=f'BIC min (k={best_k_bic})')
ax.axvline(best_k_aic, color='#378ADD', alpha=0.3, linestyle=':', label=f'AIC min (k={best_k_aic})')
ax.set_xlabel('Number of clusters (k)', fontsize=11)
ax.set_ylabel('Information criterion', fontsize=11)
ax.set_title('Gaussian Mixture Model — AIC & BIC — london-final-light', fontsize=12)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUT / '5b_gmm_criteria.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5b_gmm_criteria.png')

In [ ]:
from collections import Counter
votes = [best_k_sil, best_k_aic, best_k_bic]
vote_counts = Counter(votes)
FINAL_K = vote_counts.most_common(1)[0][0]
if vote_counts.most_common(1)[0][1] == 1:
    FINAL_K = best_k_sil
print(f'Silhouette best k: {best_k_sil}')
print(f'AIC best k:        {best_k_aic}')
print(f'BIC best k:        {best_k_bic}')
print(f'\n>>> FINAL k = {FINAL_K} <<<')

final_km, final_labels = km_models[FINAL_K]
risk['cluster_raw'] = final_labels

In [ ]:
# Re-label clusters Tier 1 (highest risk) .. Tier k (lowest) by mean risk_score_scaled
cluster_mean_risk = (risk.groupby('cluster_raw')['risk_score_scaled'].mean()
                     .sort_values(ascending=False).reset_index())
cluster_mean_risk['tier'] = range(1, FINAL_K + 1)
tier_map = dict(zip(cluster_mean_risk['cluster_raw'], cluster_mean_risk['tier']))
risk['tier'] = risk['cluster_raw'].map(tier_map)

print('Cluster -> Tier mapping (by mean risk score):')
print(cluster_mean_risk)
print('\nLSOAs per tier:')
print(risk['tier'].value_counts().sort_index())

## 5c. Cluster Profiling

In [ ]:
PROFILE_COLS = ['risk_score_scaled', 'crime_count'] + MODEL_FEATS
profile = risk.groupby('tier')[PROFILE_COLS].agg(['mean','median','std']).round(2)
print('--- Cluster Profiles ---')
print(profile.to_string())

profile_flat = risk.groupby('tier')[PROFILE_COLS].mean().round(2)
profile_flat['n_lsoas'] = risk.groupby('tier').size()
profile_flat['pct_london'] = (profile_flat['n_lsoas'] / len(risk) * 100).round(1)
profile_flat = profile_flat.reset_index()
print('\n--- Flat Profile (mean per tier) ---')
print(profile_flat.to_string(index=False))

In [ ]:
print('--- Top Boroughs per Tier ---')
for tier in sorted(risk['tier'].unique()):
    top_boros = (risk[risk['tier'] == tier].groupby('lad22nm')['risk_score_scaled']
                 .mean().sort_values(ascending=False).head(5))
    print(f'\nTier {tier}:')
    for boro, score in top_boros.items():
        n = len(risk[(risk['tier'] == tier) & (risk['lad22nm'] == boro)])
        print(f'  {boro:<25} mean_risk={score:.1f}  n_lsoas={n}')

In [ ]:
plot_cols = ['risk_score_scaled'] + MODEL_FEATS
short_labels = ['Risk\nscore'] + [f.replace('_', '\n') for f in MODEL_FEATS]
profile_norm = profile_flat.set_index('tier')[plot_cols].copy()
profile_norm = (profile_norm - profile_norm.min()) / (profile_norm.max() - profile_norm.min())

_fmt = lambda v: (f'{float(v):,.0f}' if abs(float(v))>=1000 else f'{float(v):.0f}' if abs(float(v))>=10 else f'{float(v):.1f}')
fig, axes = plt.subplots(1, FINAL_K, figsize=(4 * FINAL_K, 5), sharey=True)
if FINAL_K == 1:
    axes = [axes]
for tier, ax in zip(sorted(profile_norm.index), axes):
    colour = TIER_COLOURS.get(tier, '#888888')
    ax.bar(short_labels, profile_norm.loc[tier].values, color=colour, edgecolor='white', alpha=0.85)
    _raw = profile_flat.set_index('tier').loc[tier, plot_cols].values
    for _xi, (_nv, _rv) in enumerate(zip(profile_norm.loc[tier].values, _raw)):
        ax.text(_xi, _nv + 0.02, _fmt(_rv), ha='center', va='bottom', fontsize=6, rotation=90, clip_on=False)
    n = int(profile_flat.loc[profile_flat['tier'] == tier, 'n_lsoas'].values[0])
    pct = profile_flat.loc[profile_flat['tier'] == tier, 'pct_london'].values[0]
    ax.set_title(f'Tier {tier}\n(n={n}, {pct}%)', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.4)
    ax.tick_params(axis='x', labelsize=7)
    ax.spines[['top','right']].set_visible(False)
    if tier == 1:
        ax.set_ylabel('Normalised mean (0=low, 1=high)', fontsize=9)
plt.suptitle('Cluster profiles — london-final-light (bars normalised across tiers; labels = raw mean)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(OUT / '5c_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5c_cluster_profiles.png')

In [ ]:
sil_vals = silhouette_samples(X, final_labels)
risk['silhouette'] = sil_vals
fig, ax = plt.subplots(figsize=(8, 5))
y_lower = 10
for tier in sorted(risk['tier'].unique()):
    colour = TIER_COLOURS.get(int(tier), '#888888')
    mask = risk['tier'] == tier
    tier_sil = np.sort(sil_vals[mask.values])
    y_upper = y_lower + tier_sil.shape[0]
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, tier_sil, facecolor=colour, edgecolor=colour, alpha=0.7)
    ax.text(-0.05, y_lower + tier_sil.shape[0]/2, f'T{tier}', ha='right', va='center', fontsize=9)
    y_lower = y_upper + 10
overall_sil = silhouette_score(X, final_labels)
ax.axvline(overall_sil, color='red', linestyle='--', linewidth=1.5, label=f'Mean silhouette = {overall_sil:.3f}')
ax.set_xlabel('Silhouette coefficient', fontsize=11)
ax.set_ylabel('LSOA (grouped by tier)', fontsize=11)
ax.set_title(f'Silhouette plot — k={FINAL_K} clusters — london-final-light', fontsize=12)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUT / '5a_silhouette_detail.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5a_silhouette_detail.png')

## Section-2. Choropleth Map

In [ ]:
shp_files = list(SHP_DIR.rglob('*.shp'))
print(f'Found {len(shp_files)} shapefiles')
gdf = pd.concat([gpd.read_file(f) for f in shp_files], ignore_index=True)
gdf = gpd.GeoDataFrame(gdf, crs=gpd.read_file(shp_files[0]).crs)
print(f'GeoDataFrame shape: {gdf.shape}  CRS: {gdf.crs}')

In [ ]:
gdf_merged = gdf.merge(risk[['lsoa21cd', 'tier', 'risk_score_scaled']], on='lsoa21cd', how='left')
gdf_merged = gdf_merged.to_crs(epsg=4326)
print(f'Merged GDF shape: {gdf_merged.shape}  Unmatched: {gdf_merged["tier"].isnull().sum()}')

fig, ax = plt.subplots(figsize=(14, 10))
fallback = ['#D62728','#FF7F0E','#2CA02C','#1F77B4','#9467BD','#BCBD22']
tiers_sorted = sorted(risk['tier'].unique())
for i, tier in enumerate(tiers_sorted):
    colour = TIER_COLOURS.get(int(tier), fallback[i % len(fallback)])
    gdf_merged[gdf_merged['tier'] == tier].plot(ax=ax, color=colour, linewidth=0.05, edgecolor='white', alpha=0.85)
unmatched = gdf_merged[gdf_merged['tier'].isnull()]
if len(unmatched) > 0:
    unmatched.plot(ax=ax, color='#cccccc', linewidth=0.05, edgecolor='white')
tier_labels = {1:'Tier 1 — Critical demand',2:'Tier 2 — High demand',3:'Tier 3 — Moderate demand',
               4:'Tier 4 — Low demand',5:'Tier 5 — Minimal demand',6:'Tier 6 — Lowest demand'}
patches = [mpatches.Patch(color=TIER_COLOURS.get(int(t), fallback[i % len(fallback)]),
                          label=tier_labels.get(int(t), f'Tier {int(t)}')) for i, t in enumerate(tiers_sorted)]
ax.legend(handles=patches, loc='lower left', fontsize=10, framealpha=0.9)
ax.set_title('London LSOA Policing Demand Tiers — london-final-light', fontsize=16, fontweight='bold', pad=15)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT / '5c_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5c_map.png')

## Section-3. Save Outputs

In [ ]:
cluster_out = risk[['lsoa21cd','lsoa21nm','lad22nm','tier','risk_score_scaled','crime_count']
                   + MODEL_FEATS + ['silhouette']].copy()
cluster_out.to_parquet(OUT / 'phase5_clusters.parquet', index=False)
print(f'Saved phase5_clusters.parquet — {cluster_out.shape}')

profile_flat.to_csv(OUT / 'phase5_cluster_profiles.csv', index=False)
print(f'Saved phase5_cluster_profiles.csv — {profile_flat.shape}')

## Section-4: Forced k=4 Solution (alternative)

In [ ]:
K4 = 4
km4 = KMeans(n_clusters=K4, random_state=42, n_init=20, max_iter=500)
labels4 = km4.fit_predict(X)
sil4 = silhouette_score(X, labels4)
sil4_samples = silhouette_samples(X, labels4)

risk4 = risk[['lsoa21cd','lsoa21nm','lad22nm','crime_count'] + MODEL_FEATS +
             ['risk_score_scaled','nb_predicted','nb_ratio']].copy()
risk4['cluster_raw'] = labels4
risk4['silhouette']  = sil4_samples
cluster_mean4 = (risk4.groupby('cluster_raw')['risk_score_scaled'].mean()
                 .sort_values(ascending=False).reset_index())
cluster_mean4['tier'] = range(1, K4 + 1)
risk4['tier'] = risk4['cluster_raw'].map(dict(zip(cluster_mean4['cluster_raw'], cluster_mean4['tier'])))

print(f'k=4  silhouette={sil4:.4f}')
print('\nLSOAs per tier:')
print(risk4['tier'].value_counts().sort_index())
print('\nMean risk score per tier:')
print(risk4.groupby('tier')['risk_score_scaled'].mean().round(2))

In [ ]:
TIER4_COLOURS = {1: '#D62728', 2: '#FF7F0E', 3: '#2CA02C', 4: '#1F77B4'}
PROFILE4_COLS = ['risk_score_scaled','crime_count'] + MODEL_FEATS
profile4 = risk4.groupby('tier')[PROFILE4_COLS].mean().round(2)
profile4['n_lsoas'] = risk4.groupby('tier').size()
profile4['pct_london'] = (profile4['n_lsoas'] / len(risk4) * 100).round(1)
profile4 = profile4.reset_index()

plot_cols4 = ['risk_score_scaled'] + MODEL_FEATS
short_labels4 = ['Risk\nscore'] + [f.replace('_', '\n') for f in MODEL_FEATS]
profile4_norm = profile4.set_index('tier')[plot_cols4].copy()
profile4_norm = (profile4_norm - profile4_norm.min()) / (profile4_norm.max() - profile4_norm.min())

_fmt = lambda v: (f'{float(v):,.0f}' if abs(float(v))>=1000 else f'{float(v):.0f}' if abs(float(v))>=10 else f'{float(v):.1f}')
fig, axes = plt.subplots(1, K4, figsize=(4 * K4, 5), sharey=True)
for tier, ax in zip(range(1, K4 + 1), axes):
    ax.bar(short_labels4, profile4_norm.loc[tier].values, color=TIER4_COLOURS[tier], edgecolor='white', alpha=0.85)
    _raw = profile4.set_index('tier').loc[tier, plot_cols4].values
    for _xi, (_nv, _rv) in enumerate(zip(profile4_norm.loc[tier].values, _raw)):
        ax.text(_xi, _nv + 0.02, _fmt(_rv), ha='center', va='bottom', fontsize=6, rotation=90, clip_on=False)
    n = int(profile4.loc[profile4['tier'] == tier, 'n_lsoas'].values[0])
    pct = profile4.loc[profile4['tier'] == tier, 'pct_london'].values[0]
    ax.set_title(f'Tier {tier}\n(n={n}, {pct}%)', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.4)
    ax.tick_params(axis='x', labelsize=7)
    ax.spines[['top','right']].set_visible(False)
    if tier == 1:
        ax.set_ylabel('Normalised mean (0=low, 1=high)', fontsize=9)
plt.suptitle('k=4 Cluster profiles — london-final-light (bars normalised across tiers; labels = raw mean)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(OUT / '5d_k4_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5d_k4_cluster_profiles.png')

In [ ]:
TIER4_LABELS = {1:'Tier 1 — Critical demand',2:'Tier 2 — High demand',3:'Tier 3 — Moderate demand',4:'Tier 4 — Low demand'}
gdf_k4 = gdf.merge(risk4[['lsoa21cd','tier','risk_score_scaled']], on='lsoa21cd', how='left').to_crs(epsg=4326)
fig, ax = plt.subplots(figsize=(14, 10))
for tier in range(1, K4 + 1):
    gdf_k4[gdf_k4['tier'] == tier].plot(ax=ax, color=TIER4_COLOURS[tier], linewidth=0.05, edgecolor='white', alpha=0.85)
um4 = gdf_k4[gdf_k4['tier'].isnull()]
if len(um4) > 0:
    um4.plot(ax=ax, color='#cccccc', linewidth=0.05, edgecolor='white')
patches4 = [mpatches.Patch(color=TIER4_COLOURS[t], label=TIER4_LABELS[t]) for t in range(1, K4 + 1)]
ax.legend(handles=patches4, loc='lower left', fontsize=10, framealpha=0.9)
ax.set_title('London LSOA Policing Demand Tiers (k=4) — london-final-light', fontsize=16, fontweight='bold', pad=15)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT / '5d_k4_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 5d_k4_map.png')

In [ ]:
risk4[['lsoa21cd','lsoa21nm','lad22nm','tier','risk_score_scaled','crime_count']
      + MODEL_FEATS + ['silhouette']].to_parquet(OUT / 'phase5_clusters_k4.parquet', index=False)
print('Saved phase5_clusters_k4.parquet')
profile4.to_csv(OUT / 'phase5_cluster_profiles_k4.csv', index=False)
print('Saved phase5_cluster_profiles_k4.csv')

In [ ]:
print('=' * 55)
print('PHASE 5 SUMMARY (london-final-light)')
print('=' * 55)
print(f'Total LSOAs:               {len(risk)}')
print(f'Clustering features:       {CLUSTER_FEATURES}')
print(f'Final k (tiers):           {FINAL_K}')
print(f'Silhouette score (k={FINAL_K}):   {overall_sil:.4f}')
print(f'Best k — silhouette/AIC/BIC: {best_k_sil}/{best_k_aic}/{best_k_bic}')
print('\nLSOAs per tier:')
for _, row in profile_flat.iterrows():
    print(f"  Tier {int(row['tier'])}: {int(row['n_lsoas'])} LSOAs ({row['pct_london']}%)  mean_risk={row['risk_score_scaled']:.1f}")
print('=' * 55)
print(f'Outputs saved to: {OUT}')